# Albanian NER — hand annotation

Per-token BIO dropdown UI over `data/raw/wiki_segmented.jsonl`. Writes one JSON line per saved sentence to `data/labeled/sq_hand100.jsonl`. Already-labeled IDs are skipped, so closing and reopening the notebook resumes where you left off.

Tagset: `O`, `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`.

Tips:
- **B-X** starts an entity. **I-X** continues the same entity. If two entities of the same type touch, the second one starts a new **B-X**.
- If a sentence is unusable (HTML cruft, foreign-language fragment, no entities you can verify), hit **Skip** — the sentence will not be saved and won't reappear.
- **Back** undoes the last save (re-opens the previous sentence).

In [ ]:
import json
from pathlib import Path

import ipywidgets as W
from IPython.display import display

REPO_ROOT = Path.cwd().parent
CANDIDATES = REPO_ROOT / "data/raw/wiki_segmented.jsonl"
LABELED = REPO_ROOT / "data/labeled/sq_hand100.jsonl"
TARGET_N = 100

TAGS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]

assert CANDIDATES.exists(), (
    f"Run `uv run python -m src.data.wiki_sample ...` and `uv run python -m src.data.segment ...` first\n"
    f"(expected {CANDIDATES})"
)
LABELED.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_candidates():
    return [json.loads(line) for line in CANDIDATES.read_text(encoding="utf-8").splitlines() if line.strip()]


def load_labeled_ids():
    if not LABELED.exists():
        return set()
    ids = set()
    for line in LABELED.read_text(encoding="utf-8").splitlines():
        if line.strip():
            ids.add(json.loads(line)["id"])
    return ids


def append_record(rec):
    with LABELED.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def drop_last_record():
    """Remove the last line of LABELED (used by Back)."""
    if not LABELED.exists():
        return None
    lines = LABELED.read_text(encoding="utf-8").splitlines()
    if not lines:
        return None
    last = json.loads(lines[-1])
    LABELED.write_text("\n".join(lines[:-1]) + ("\n" if len(lines) > 1 else ""), encoding="utf-8")
    return last["id"]

In [ ]:
candidates = load_candidates()
labeled_ids = load_labeled_ids()
queue = [c for c in candidates if c["id"] not in labeled_ids]
skipped_ids = set()

print(f"{len(candidates)} candidate sentences, {len(labeled_ids)} already labeled, {len(queue)} remaining")

In [ ]:
header = W.HTML()
context = W.HTML()
token_box = W.VBox()
save_btn = W.Button(description="Save", button_style="success")
skip_btn = W.Button(description="Skip", button_style="warning")
back_btn = W.Button(description="Back", button_style="")
status = W.HTML()

ui = W.VBox([header, context, token_box, W.HBox([save_btn, skip_btn, back_btn]), status])

state = {"current": None, "dropdowns": []}


def next_record():
    while queue:
        rec = queue.pop(0)
        if rec["id"] in labeled_ids or rec["id"] in skipped_ids:
            continue
        return rec
    return None


def render(rec):
    state["current"] = rec
    state["dropdowns"] = []
    if rec is None:
        header.value = "<h3>All done.</h3>"
        context.value = ""
        token_box.children = []
        return
    done = len(labeled_ids)
    header.value = (
        f"<h3>{rec['id']} &mdash; {done}/{TARGET_N} labeled, {len(queue)} remaining</h3>"
    )
    context.value = (
        f"<p><i>{rec.get('title', '')}</i> &mdash; "
        f"<a href=\"{rec.get('source_url', '#')}\" target=\"_blank\">source</a></p>"
        f"<p style=\"font-size:1.05em\">{rec['text']}</p>"
    )
    rows = []
    for tok in rec["tokens"]:
        dd = W.Dropdown(options=TAGS, value="O", layout=W.Layout(width="110px"))
        label = W.HTML(f"<code style='font-size:1.05em'>{tok}</code>")
        rows.append(W.HBox([dd, label], layout=W.Layout(align_items="center")))
        state["dropdowns"].append(dd)
    token_box.children = rows
    status.value = ""


def validate_bio(tags):
    last_type = None
    for t in tags:
        if t == "O":
            last_type = None
            continue
        prefix, etype = t.split("-", 1)
        if prefix == "B":
            last_type = etype
        elif prefix == "I":
            if last_type != etype:
                return f"`{t}` without preceding `B-{etype}`"
            last_type = etype
    return None


def on_save(_):
    rec = state["current"]
    if rec is None:
        return
    tags = [dd.value for dd in state["dropdowns"]]
    err = validate_bio(tags)
    if err:
        status.value = f"<span style='color:#b00'>Invalid BIO: {err}</span>"
        return
    out = {
        "id": rec["id"],
        "tokens": rec["tokens"],
        "ner_tags": tags,
        "text": rec["text"],
        "source_url": rec.get("source_url"),
        "title": rec.get("title"),
    }
    append_record(out)
    labeled_ids.add(rec["id"])
    if len(labeled_ids) >= TARGET_N:
        render(None)
        status.value = f"<b>Hit target of {TARGET_N}.</b>"
        return
    render(next_record())


def on_skip(_):
    rec = state["current"]
    if rec is None:
        return
    skipped_ids.add(rec["id"])
    render(next_record())


def on_back(_):
    last_id = drop_last_record()
    if last_id is None:
        status.value = "Nothing to undo."
        return
    labeled_ids.discard(last_id)
    # Re-queue the undone sentence at the front, plus the current one.
    prev = next((c for c in candidates if c["id"] == last_id), None)
    if state["current"] is not None:
        queue.insert(0, state["current"])
    if prev is not None:
        queue.insert(0, prev)
    render(next_record())


save_btn.on_click(on_save)
skip_btn.on_click(on_skip)
back_btn.on_click(on_back)

render(next_record())
display(ui)

## Quick stats on what's been labeled

Re-run the cell below at any point to check progress / class balance.

In [ ]:
from collections import Counter

if LABELED.exists():
    recs = [json.loads(line) for line in LABELED.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"{len(recs)} labeled sentences")
    ent = Counter()
    for r in recs:
        for tag in r["ner_tags"]:
            if tag.startswith("B-"):
                ent[tag[2:]] += 1
    print("entity counts:", dict(ent))
else:
    print("no labels yet")